In [35]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from utils import simulate_bagging_and_ijk_var_calculation

In [36]:
####### Simulation parameters  #########
n_x = 1_00 
n_sim = 100
B = 1000
portion_non_zero_weights = 1.

var_x = 0.5
seed = 42
m = int(n_x*portion_non_zero_weights)
folder_name = "weighted_bagged_mean_estimator"
max_workers = 8

# Data for Simulation
rng = np.random.default_rng(seed)
x_sim = rng.normal(0,var_x**0.5,(n_sim, n_x))
weights = np.zeros(n_x)
weights[:m] = 1/m

# True Variance
true_var = n_x/m**2 * var_x


In [38]:
theta_bagged= np.zeros(n_sim)
theta_bagged_var_ijk = np.zeros(n_sim)

# run simulations
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(
            simulate_bagging_and_ijk_var_calculation,
            x1=x_sim[i],
            B=B,
            sim_i=i,
            seed=seed,
            weights=weights,
            m=m
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _theta_bagged_var_ijk, _theta_bagged = future.result()
        theta_bagged[i] =_theta_bagged
        theta_bagged_var_ijk[i] = _theta_bagged_var_ijk
       
print(f"\nMean of IJK_var_est of bagged_mean_estimator: {round(theta_bagged_var_ijk.mean(),10)}")
print(f"True variance of bagged_mean_estimator: {true_var}\n")

print(f"Mean of bagged_mean_estimator: {round(theta_bagged.mean(),10)}")
print(f"True mean of bagged_mean_estimator: {0}\n")

Simulations: 100%|██████████| 100/100 [00:00<00:00, 200.26simulation/s]



Mean of IJK_var_est of bagged_mean_estimator: 0.0049793172
True variance of bagged_mean_estimator: 0.005

Mean of bagged_mean_estimator: -0.0073471949
True mean of bagged_mean_estimator: 0

